In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
base_path = Path("..")

processed_path = base_path / "data" / "processed"

prices = pd.read_csv(processed_path / "prices_clean.csv")

prices["date"] = pd.to_datetime(prices["date"])

prices = prices.sort_values(["ticker","date"]).reset_index(drop=True)

prices.head()

,date,ticker,open,high,low,close,adj_close,volume
0,2018-01-02,AKAM,65.129997,65.940002,64.699997,65.559998,65.559998,1425900.0
1,2018-01-03,AKAM,65.269997,66.000000,65.099998,65.940002,65.940002,2287700.0
2,2018-01-04,AKAM,66.160004,66.250000,65.440002,65.599998,65.599998,1870600.0
3,2018-01-05,AKAM,65.699997,65.860001,65.459999,65.830002,65.830002,1056000.0
4,2018-01-08,AKAM,65.949997,66.089996,65.209999,65.879997,65.879997,1185900.0


In [3]:
prices["ret_1d"] = prices.groupby("ticker")["adj_close"].pct_change()

prices.head()

C:\Users\nch\AppData\Local\Temp\ipykernel_3516\2971684174.py:1: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  prices["ret_1d"] = prices.groupby("ticker")["adj_close"].pct_change()


,date,ticker,open,high,low,close,adj_close,volume,ret_1d
0,2018-01-02,AKAM,65.129997,65.940002,64.699997,65.559998,65.559998,1425900.0,NaN
1,2018-01-03,AKAM,65.269997,66.000000,65.099998,65.940002,65.940002,2287700.0,0.005796
2,2018-01-04,AKAM,66.160004,66.250000,65.440002,65.599998,65.599998,1870600.0,-0.005156
3,2018-01-05,AKAM,65.699997,65.860001,65.459999,65.830002,65.830002,1056000.0,0.003506
4,2018-01-08,AKAM,65.949997,66.089996,65.209999,65.879997,65.879997,1185900.0,0.000759


In [4]:
prices["mom_1m"] = prices.groupby("ticker")["adj_close"].pct_change(21)

prices["mom_3m"] = prices.groupby("ticker")["adj_close"].pct_change(63)

prices["mom_6m"] = prices.groupby("ticker")["adj_close"].pct_change(126)

C:\Users\nch\AppData\Local\Temp\ipykernel_3516\3487094632.py:1: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  prices["mom_1m"] = prices.groupby("ticker")["adj_close"].pct_change(21)
C:\Users\nch\AppData\Local\Temp\ipykernel_3516\3487094632.py:3: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  prices["mom_3m"] = prices.groupby("ticker")["adj_close"].pct_change(63)
C:\Users\nch\AppData\Local\Temp\ipykernel_3516\3487094632.py:5: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-lea

In [5]:
prices["vol_1m"] = prices.groupby("ticker")["ret_1d"].transform(lambda x: x.rolling(21).std())

prices["vol_3m"] = prices.groupby("ticker")["ret_1d"].transform(lambda x: x.rolling(63).std())

prices["vol_6m"] = prices.groupby("ticker")["ret_1d"].transform(lambda x: x.rolling(126).std())

In [6]:
prices["avg_volume_1m"] = prices.groupby("ticker")["volume"].transform(lambda x: x.rolling(21).mean())

prices["avg_volume_3m"] = prices.groupby("ticker")["volume"].transform(lambda x: x.rolling(63).mean())

prices["volume_ratio"] = prices["volume"] / prices["avg_volume_3m"]

In [7]:
prices["fwd_ret_1m"] = prices.groupby("ticker")["adj_close"].shift(-21) / prices["adj_close"] - 1

In [8]:
prices.to_csv(processed_path / "factor_data_daily.csv", index=False)

print("Saved factor dataset.")

Saved factor dataset.


In [9]:
prices.head()

,date,ticker,open,high,low,close,adj_close,volume,ret_1d,mom_1m,mom_3m,mom_6m,vol_1m,vol_3m,vol_6m,avg_volume_1m,avg_volume_3m,volume_ratio,fwd_ret_1m
0,2018-01-02,AKAM,65.129997,65.940002,64.699997,65.559998,65.559998,1425900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030506
1,2018-01-03,AKAM,65.269997,66.000000,65.099998,65.940002,65.940002,2287700.0,0.005796,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.003336
2,2018-01-04,AKAM,66.160004,66.250000,65.440002,65.599998,65.599998,1870600.0,-0.005156,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.042530
3,2018-01-05,AKAM,65.699997,65.860001,65.459999,65.830002,65.830002,1056000.0,0.003506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.032812
4,2018-01-08,AKAM,65.949997,66.089996,65.209999,65.879997,65.879997,1185900.0,0.000759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.021251


In [10]:
prices["month"] = prices["date"].dt.to_period("M")

prices.head()

,date,ticker,open,high,low,close,adj_close,volume,ret_1d,mom_1m,mom_3m,mom_6m,vol_1m,vol_3m,vol_6m,avg_volume_1m,avg_volume_3m,volume_ratio,fwd_ret_1m,month
0,2018-01-02,AKAM,65.129997,65.940002,64.699997,65.559998,65.559998,1425900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030506,2018-01
1,2018-01-03,AKAM,65.269997,66.000000,65.099998,65.940002,65.940002,2287700.0,0.005796,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.003336,2018-01
2,2018-01-04,AKAM,66.160004,66.250000,65.440002,65.599998,65.599998,1870600.0,-0.005156,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.042530,2018-01
3,2018-01-05,AKAM,65.699997,65.860001,65.459999,65.830002,65.830002,1056000.0,0.003506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.032812,2018-01
4,2018-01-08,AKAM,65.949997,66.089996,65.209999,65.879997,65.879997,1185900.0,0.000759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.021251,2018-01


In [11]:
monthly = prices.groupby(["ticker","month"]).tail(1).copy()

monthly["month"] = monthly["month"].astype(str)

monthly.head()

,date,ticker,open,high,low,close,adj_close,volume,ret_1d,mom_1m,mom_3m,mom_6m,vol_1m,vol_3m,vol_6m,avg_volume_1m,avg_volume_3m,volume_ratio,fwd_ret_1m,month
20,2018-01-31,AKAM,67.000000,68.300003,66.589996,66.989998,66.989998,1965900.0,0.006460,NaN,NaN,NaN,NaN,NaN,NaN,1.770567e+06,NaN,NaN,0.036423,2018-01
39,2018-02-28,AKAM,68.699997,69.000000,67.459999,67.459999,67.459999,1040200.0,-0.011430,0.003123,NaN,NaN,0.019263,NaN,NaN,2.086043e+06,NaN,NaN,0.052179,2018-02
60,2018-03-29,AKAM,70.820000,71.519997,70.029999,70.980003,70.980003,2001100.0,0.013276,0.052179,NaN,NaN,0.022624,NaN,NaN,1.846005e+06,NaN,NaN,0.009439,2018-03
81,2018-04-30,AKAM,71.510002,72.220001,71.050003,71.650002,71.650002,3787800.0,0.007736,0.009439,0.065428,NaN,0.016202,0.019251,NaN,1.210262e+06,1.714103e+06,2.209785,0.061828,2018-04
103,2018-05-31,AKAM,76.019997,76.019997,75.000000,75.379997,75.379997,2683700.0,-0.009201,0.064088,0.109835,NaN,0.015120,0.018089,NaN,1.964238e+06,1.728498e+06,1.552619,-0.028522,2018-05


In [12]:
print(monthly.shape)
print(monthly["ticker"].nunique())
print(monthly["month"].nunique())

(1824, 20)
19
96


In [13]:
model_data = monthly[
    [
        "date",
        "month",
        "ticker",
        "mom_1m",
        "mom_3m",
        "mom_6m",
        "vol_1m",
        "vol_3m",
        "vol_6m",
        "volume_ratio",
        "fwd_ret_1m"
    ]
].dropna().reset_index(drop=True)

model_data.head()

,date,month,ticker,mom_1m,mom_3m,mom_6m,vol_1m,vol_3m,vol_6m,volume_ratio,fwd_ret_1m
0,2018-07-31,2018-07,AKAM,0.027721,0.062394,0.130709,0.015373,0.015739,0.017518,1.124732,-0.000399
1,2018-08-31,2018-08,AKAM,0.011578,-0.025042,0.076350,0.012528,0.016673,0.017299,0.562483,-0.076258
2,2018-09-28,2018-09,AKAM,-0.027649,-0.001092,0.056318,0.010474,0.014896,0.015277,1.116898,-0.151743
3,2018-10-31,2018-10,AKAM,0.040916,-0.027329,0.011197,0.043047,0.026603,0.022424,2.227639,-0.048443
4,2018-11-30,2018-11,AKAM,-0.048443,-0.085041,-0.107954,0.018436,0.027697,0.022771,0.885031,-0.160727


In [14]:
print(model_data.shape)

print(model_data["ticker"].nunique())

print(model_data["month"].nunique())

(1572, 11)
19
89


In [15]:
model_data.to_csv(processed_path / "model_data_monthly.csv", index=False)

print("Saved monthly modeling dataset.")

Saved monthly modeling dataset.
